# 🧠 Conditional Variational Autoencoder (CVAE) — Image Generation

This notebook trains a **Conditional VAE (CVAE)** on a manually collected image dataset.
After training, it generates new images conditioned on a specific class label,
with an **interactive slider** to control the number of generated images.

### 📌 Pipeline Overview
1. Mount Google Drive & load dataset
2. Preprocess images and labels
3. Build the CVAE architecture (Encoder → Sampling → Decoder)
4. Train the CVAE
5. Visualize reconstructed images with a slider
6. Generate new images conditioned on class label with a slider

## 📦 Section 1 — Install & Import Dependencies

In [ ]:
# Install required packages
!pip install ipywidgets --quiet

import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers, backend as K
from tensorflow.keras.preprocessing.image import img_to_array, load_img

from sklearn.model_selection import train_test_split

import ipywidgets as widgets
from IPython.display import display, clear_output

print(f'TensorFlow version: {tf.__version__}')
print(f'Keras version:      {keras.__version__}')
print(f'GPU available:      {len(tf.config.list_physical_devices("GPU")) > 0}')

## 📁 Section 2 — Mount Google Drive & Set Dataset Path

Upload your dataset to Google Drive before running this cell.

**Expected folder structure:**
```
Dataset/
├── 0/          ← Class 0 images (e.g. Person A / Object A)
│   ├── img1.jpg
│   └── ...
└── 1/          ← Class 1 images (e.g. Person B / Object B)
    ├── img1.jpg
    └── ...
```
> ⚠️ Add more subfolders (e.g. `2/`, `3/`) for more classes — update `NUM_CLASSES` below accordingly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ✏️ Update this path to where your Dataset folder lives in Drive
DATASET_PATH = '/content/drive/MyDrive/Dataset'

print(f'Dataset path set to: {DATASET_PATH}')
print('Classes found:', sorted(os.listdir(DATASET_PATH)))

## 🖼️ Section 3 — Configuration & Hyperparameters

In [ ]:
# ─── Image Settings ───────────────────────────────────────────────────────────
IMG_SIZE       = (100, 100)       # Resize all images to this resolution
IMG_CHANNELS   = 3                # RGB
ORIGINAL_DIM   = IMG_SIZE[0] * IMG_SIZE[1] * IMG_CHANNELS  # Flattened size

# ─── Dataset Settings ─────────────────────────────────────────────────────────
NUM_CLASSES    = 2                # ✏️ Change if you have more classes
ROTATE_CLASS   = '0'              # Class whose images need -90° rotation fix
ROTATE_ANGLE   = -90
TEST_SPLIT     = 0.2

# ─── Model Architecture ───────────────────────────────────────────────────────
INTERMEDIATE_DIM = 512            # Dense hidden layer size
LATENT_DIM       = 128            # Size of the latent space z

# ─── Training Settings ────────────────────────────────────────────────────────
EPOCHS         = 50
BATCH_SIZE     = 32
LEARNING_RATE  = 1e-4

print('Configuration loaded.')
print(f'  Image size:      {IMG_SIZE} → flattened dim = {ORIGINAL_DIM}')
print(f'  Num classes:     {NUM_CLASSES}')
print(f'  Latent dim:      {LATENT_DIM}')
print(f'  Epochs:          {EPOCHS}')

## 📂 Section 4 — Load & Preprocess Dataset

In [ ]:
def load_images(folder_path, target_size=IMG_SIZE, rotate_class=ROTATE_CLASS, rotate_angle=ROTATE_ANGLE):
    """
    Load images from a folder structured as:
        folder_path/
            class_name/
                image_file.jpg

    Applies optional rotation correction for a specific class.
    Returns:
        images : np.ndarray of shape (N, H, W, C), float32 in [0, 1]
        labels : np.ndarray of shape (N,), integer class indices
    """
    images = []
    labels = []

    # Sort to ensure consistent label ordering across runs
    class_names = sorted(os.listdir(folder_path))
    label_mapping = {name: idx for idx, name in enumerate(class_names)}
    print(f'Label mapping: {label_mapping}')

    for class_name in class_names:
        class_path = os.path.join(folder_path, class_name)
        if not os.path.isdir(class_path):
            continue

        label_value = label_mapping[class_name]
        count = 0

        for filename in os.listdir(class_path):
            img_path = os.path.join(class_path, filename)
            try:
                img = load_img(img_path, target_size=target_size)
                # Apply rotation fix if needed
                if class_name == rotate_class:
                    img = img.rotate(rotate_angle)
                img_array = img_to_array(img) / 255.0  # Normalize to [0, 1]
                images.append(img_array)
                labels.append(label_value)
                count += 1
            except Exception as e:
                print(f'  ⚠️  Skipped {filename}: {e}')

        print(f'  Class "{class_name}" (label {label_value}): {count} images loaded')

    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)


# ─── Load ─────────────────────────────────────────────────────────────────────
images, labels = load_images(DATASET_PATH)

print(f'\nTotal images loaded: {len(images)}')
print(f'Images shape:        {images.shape}')
print(f'Labels distribution: { {i: int(np.sum(labels == i)) for i in range(NUM_CLASSES)} }')

# ─── Train / Test Split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=TEST_SPLIT, shuffle=True, stratify=labels, random_state=42
)

print(f'\nTrain samples: {len(X_train)}')
print(f'Test  samples: {len(X_test)}')

## 👁️ Section 5 — Visualize Sample Images from Dataset

In [ ]:
def show_dataset_samples(images, labels, num_classes=NUM_CLASSES, samples_per_class=5):
    """Display a grid of sample images from each class."""
    fig, axes = plt.subplots(
        num_classes, samples_per_class,
        figsize=(samples_per_class * 2.5, num_classes * 2.5)
    )
    fig.suptitle('Dataset Samples per Class', fontsize=14, fontweight='bold', y=1.01)

    for cls in range(num_classes):
        class_images = images[labels == cls]
        for j in range(samples_per_class):
            ax = axes[cls, j] if num_classes > 1 else axes[j]
            if j < len(class_images):
                ax.imshow(class_images[j])
            else:
                ax.axis('off')
            ax.set_xticks([])
            ax.set_yticks([])
            if j == 0:
                ax.set_ylabel(f'Class {cls}', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.show()

show_dataset_samples(images, labels)

## 🏗️ Section 6 — Build the CVAE Architecture

The **Conditional VAE** architecture:

```
Input Image  ──┐
               ├─► Concatenate ─► Encoder ─► z_mean, z_log_var
Class Label  ──┘                                    │
                                              Reparameterize
                                                    │ z
                                              Concatenate ◄── Class Label
                                                    │
                                                 Decoder
                                                    │
                                           Reconstructed Image
```

In [ ]:
# ─── Sampling Layer (Reparameterization Trick) ─────────────────────────────────
class Sampling(layers.Layer):
    """
    Reparameterization trick: z = z_mean + eps * exp(0.5 * z_log_var)
    where eps ~ N(0, I). This allows gradients to flow through the sampling.
    """
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch   = tf.shape(z_mean)[0]
        dim     = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim), mean=0.0, stddev=1.0)
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon


# ─── Encoder ──────────────────────────────────────────────────────────────────
def build_encoder(original_dim, num_classes, intermediate_dim, latent_dim):
    """
    Encoder: (flattened_image, one_hot_label) → (z_mean, z_log_var, z)
    """
    img_input   = keras.Input(shape=(original_dim,),  name='encoder_image_input')
    label_input = keras.Input(shape=(num_classes,),   name='encoder_label_input')

    x = layers.Concatenate(name='encoder_concat')([img_input, label_input])
    x = layers.Dense(intermediate_dim, activation='relu', name='encoder_dense1')(x)
    x = layers.Dense(intermediate_dim // 2, activation='relu', name='encoder_dense2')(x)

    z_mean    = layers.Dense(latent_dim, name='z_mean')(x)
    z_log_var = layers.Dense(latent_dim, name='z_log_var')(x)
    z         = Sampling(name='z_sampling')([z_mean, z_log_var])

    encoder = keras.Model(
        inputs  = [img_input, label_input],
        outputs = [z_mean, z_log_var, z],
        name    = 'encoder'
    )
    return encoder


# ─── Decoder ──────────────────────────────────────────────────────────────────
def build_decoder(original_dim, num_classes, intermediate_dim, latent_dim):
    """
    Decoder: (z, one_hot_label) → reconstructed_image
    """
    z_input     = keras.Input(shape=(latent_dim,),  name='decoder_z_input')
    label_input = keras.Input(shape=(num_classes,), name='decoder_label_input')

    x = layers.Concatenate(name='decoder_concat')([z_input, label_input])
    x = layers.Dense(intermediate_dim // 2, activation='relu', name='decoder_dense1')(x)
    x = layers.Dense(intermediate_dim,      activation='relu', name='decoder_dense2')(x)
    output = layers.Dense(original_dim, activation='sigmoid', name='decoder_output')(x)

    decoder = keras.Model(
        inputs  = [z_input, label_input],
        outputs = output,
        name    = 'decoder'
    )
    return decoder


# ─── Full CVAE Model ──────────────────────────────────────────────────────────
class CVAE(keras.Model):
    """
    Conditional Variational Autoencoder.
    Loss = Reconstruction Loss (MSE) + KL Divergence
    """
    def __init__(self, encoder, decoder, original_dim, **kwargs):
        super().__init__(**kwargs)
        self.encoder       = encoder
        self.decoder       = decoder
        self.original_dim  = original_dim

        # Metrics to track
        self.total_loss_tracker    = keras.metrics.Mean(name='total_loss')
        self.recon_loss_tracker    = keras.metrics.Mean(name='recon_loss')
        self.kl_loss_tracker       = keras.metrics.Mean(name='kl_loss')

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        (x_flat, y_onehot), _ = data   # x_flat: images, y_onehot: class labels

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder([x_flat, y_onehot])
            reconstruction       = self.decoder([z, y_onehot])

            # Reconstruction loss (MSE scaled by image dim)
            recon_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.mean_squared_error(x_flat, reconstruction)
                )
            ) * self.original_dim

            # KL divergence: -0.5 * sum(1 + log_var - mean^2 - exp(log_var))
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
            )

            total_loss = recon_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)

        return {
            'total_loss': self.total_loss_tracker.result(),
            'recon_loss': self.recon_loss_tracker.result(),
            'kl_loss':    self.kl_loss_tracker.result(),
        }

    def call(self, inputs):
        x_flat, y_onehot             = inputs
        z_mean, z_log_var, z         = self.encoder([x_flat, y_onehot])
        reconstruction               = self.decoder([z, y_onehot])
        return reconstruction


# ─── Instantiate ──────────────────────────────────────────────────────────────
encoder = build_encoder(ORIGINAL_DIM, NUM_CLASSES, INTERMEDIATE_DIM, LATENT_DIM)
decoder = build_decoder(ORIGINAL_DIM, NUM_CLASSES, INTERMEDIATE_DIM, LATENT_DIM)
cvae    = CVAE(encoder, decoder, ORIGINAL_DIM, name='cvae')

cvae.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE))

print('\n── Encoder Summary ──')
encoder.summary()
print('\n── Decoder Summary ──')
decoder.summary()

## 🏋️ Section 7 — Train the CVAE

In [ ]:
# ─── Prepare Data ─────────────────────────────────────────────────────────────
# Flatten images: (N, H, W, C) → (N, H*W*C)
X_train_flat = X_train.reshape((len(X_train), ORIGINAL_DIM))
X_test_flat  = X_test.reshape((len(X_test),  ORIGINAL_DIM))

# One-hot encode labels: (N,) → (N, NUM_CLASSES)
Y_train_oh = keras.utils.to_categorical(y_train, NUM_CLASSES)
Y_test_oh  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

# Build tf.data dataset — input is (x, y), target is x (ignored, loss is custom)
train_dataset = tf.data.Dataset.from_tensor_slices(
    ((X_train_flat, Y_train_oh), X_train_flat)
).shuffle(buffer_size=1024).batch(BATCH_SIZE)

# ─── Callbacks ────────────────────────────────────────────────────────────────
callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='total_loss', factor=0.5, patience=5,
        min_lr=1e-6, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='total_loss', patience=10, restore_best_weights=True, verbose=1
    ),
]

# ─── Train ────────────────────────────────────────────────────────────────────
print(f'Training CVAE for up to {EPOCHS} epochs...')
history = cvae.fit(
    train_dataset,
    epochs    = EPOCHS,
    callbacks = callbacks,
)
print('\nTraining complete ✅')

## 📉 Section 8 — Plot Training Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('CVAE Training Losses', fontsize=14, fontweight='bold')

metrics = ['total_loss', 'recon_loss', 'kl_loss']
titles  = ['Total Loss', 'Reconstruction Loss', 'KL Divergence']
colors  = ['#2196F3', '#4CAF50', '#FF5722']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    ax.plot(history.history[metric], color=color, linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 🔁 Section 9 — Reconstruct Test Images with Interactive Slider

Use the slider to control **how many test images** to display alongside their reconstructions.

In [ ]:
# Pre-compute encoder outputs for the test set
z_mean_out, z_log_var_out, z_out = encoder.predict(
    [X_test_flat, Y_test_oh], verbose=0
)

MAX_DISPLAY = min(10, len(X_test_flat))

out_recon = widgets.Output()

def show_reconstructions(num_images):
    with out_recon:
        clear_output(wait=True)

        n      = int(num_images)
        z_sel  = np.concatenate([z_out[:n], Y_test_oh[:n]], axis=1)

        # NOTE: decoder takes [z, label] as separate inputs — NOT concatenated manually.
        recon  = decoder.predict([z_out[:n], Y_test_oh[:n]], verbose=0)

        fig, axes = plt.subplots(2, n, figsize=(n * 2.2, 5))
        fig.suptitle(
            f'Reconstructed Images  (n={n})',
            fontsize=13, fontweight='bold'
        )

        for i in range(n):
            col_orig  = axes[0, i] if n > 1 else axes[0]
            col_recon = axes[1, i] if n > 1 else axes[1]

            col_orig.imshow(X_test_flat[i].reshape(IMG_SIZE[0], IMG_SIZE[1], IMG_CHANNELS))
            col_orig.set_title(f'C{int(y_test[i])}', fontsize=9)
            col_orig.axis('off')

            col_recon.imshow(
                np.clip(recon[i].reshape(IMG_SIZE[0], IMG_SIZE[1], IMG_CHANNELS), 0, 1)
            )
            col_recon.axis('off')

        # Row labels
        (axes[0, 0] if n > 1 else axes[0]).set_ylabel('Original',      fontsize=10, fontweight='bold')
        (axes[1, 0] if n > 1 else axes[1]).set_ylabel('Reconstructed', fontsize=10, fontweight='bold')

        plt.tight_layout()
        plt.show()

# ─── Slider Widget ────────────────────────────────────────────────────────────
slider_recon = widgets.IntSlider(
    value=5, min=1, max=MAX_DISPLAY, step=1,
    description='# Images:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

ui_recon = widgets.VBox([
    widgets.HTML('<h3 style="margin:0">🔁 Reconstruction Viewer</h3>'),
    slider_recon,
    out_recon
])

widgets.interactive_output(show_reconstructions, {'num_images': slider_recon})
display(ui_recon)
show_reconstructions(5)

## ✨ Section 10 — Generate New Images (CVAE Conditional Generation)

Generate **brand-new images** by sampling random vectors from the latent space,
conditioned on a chosen class label.

- **Class Selector** → choose which class to generate
- **# Images Slider** → how many images to generate

In [ ]:
out_gen = widgets.Output()

def generate_images(num_images, class_label):
    with out_gen:
        clear_output(wait=True)

        n   = int(num_images)
        cls = int(class_label)

        # Sample random latent vectors from N(0, I)
        z_samples = np.random.normal(size=(n, LATENT_DIM)).astype(np.float32)

        # Condition on chosen class label
        class_oh  = np.tile(
            keras.utils.to_categorical([cls], NUM_CLASSES),
            (n, 1)
        ).astype(np.float32)

        # Decode
        generated = decoder.predict([z_samples, class_oh], verbose=0)

        # Display
        fig, axes = plt.subplots(1, n, figsize=(n * 2.5, 3))
        if n == 1:
            axes = [axes]

        fig.suptitle(
            f'Generated Images — Class {cls}   (n={n})',
            fontsize=13, fontweight='bold'
        )

        for i, ax in enumerate(axes):
            img = np.clip(
                generated[i].reshape(IMG_SIZE[0], IMG_SIZE[1], IMG_CHANNELS), 0, 1
            )
            ax.imshow(img)
            ax.set_title(f'Gen {i+1}', fontsize=9)
            ax.axis('off')

        plt.tight_layout()
        plt.show()


# ─── Widgets ──────────────────────────────────────────────────────────────────
slider_gen = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description='# Images:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

class_selector = widgets.Dropdown(
    options    = [(f'Class {i}', i) for i in range(NUM_CLASSES)],
    value      = 0,
    description= 'Class:',
    style      = {'description_width': 'initial'},
    layout     = widgets.Layout(width='300px')
)

regen_button = widgets.Button(
    description = '🎲 Regenerate',
    button_style= 'primary',
    layout      = widgets.Layout(width='150px')
)

def on_regen_click(_):
    generate_images(slider_gen.value, class_selector.value)

regen_button.on_click(on_regen_click)

controls = widgets.HBox([class_selector, slider_gen, regen_button])
ui_gen   = widgets.VBox([
    widgets.HTML('<h3 style="margin:0">✨ Conditional Image Generator</h3>'),
    controls,
    out_gen
])

widgets.interactive_output(
    generate_images,
    {'num_images': slider_gen, 'class_label': class_selector}
)
display(ui_gen)
generate_images(5, 0)

## 🗺️ Section 11 — Visualize the 2D Latent Space (PCA Projection)

Projects the high-dimensional latent vectors onto 2D using PCA
to verify class separation in the learned latent space.

In [ ]:
from sklearn.decomposition import PCA

# Encode all test images
z_mean_all, _, _ = encoder.predict([X_test_flat, Y_test_oh], verbose=0)

# Reduce to 2D
pca = PCA(n_components=2)
z_2d = pca.fit_transform(z_mean_all)

fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.Set1(np.linspace(0, 0.8, NUM_CLASSES))

for cls in range(NUM_CLASSES):
    mask = y_test == cls
    ax.scatter(
        z_2d[mask, 0], z_2d[mask, 1],
        c=[colors[cls]], label=f'Class {cls}',
        alpha=0.7, s=60, edgecolors='white', linewidths=0.5
    )

ax.set_title('Latent Space — PCA Projection (z_mean)', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

## 💾 Section 12 — Save Trained Models to Google Drive

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/CVAE_Models'
os.makedirs(SAVE_DIR, exist_ok=True)

encoder.save(os.path.join(SAVE_DIR, 'cvae_encoder.h5'))
decoder.save(os.path.join(SAVE_DIR, 'cvae_decoder.h5'))

print(f'Models saved to: {SAVE_DIR}')
print('  cvae_encoder.h5')
print('  cvae_decoder.h5')

## 🔄 Section 13 — (Optional) Load Saved Models

Run this cell to reload your models in a fresh session without retraining.

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/CVAE_Models'

encoder = keras.models.load_model(
    os.path.join(SAVE_DIR, 'cvae_encoder.h5'),
    custom_objects={'Sampling': Sampling}
)
decoder = keras.models.load_model(os.path.join(SAVE_DIR, 'cvae_decoder.h5'))

print('Models loaded successfully ✅')